In [ ]:
import pandas as pd
import numpy as np
import re
import datetime as dt
from bs4 import BeautifulSoup as bs
import requests
import json
import time
from tqdm import tqdm
import random
import math

In [ ]:
# 서포터 df 불러오기
supporter_df = pd.read_csv('20221120_Wadiz_AllProjects_Supporters_df.csv')
supporter_df

In [ ]:
# 전체 프로젝트 campaignId 불러오기
projects_df = pd.read_csv('20221120_Wadiz_corpInfo_AllProjects_dropduplicates_df.csv')
campaignId = list(projects_df['campaignId'])
totalBackedAmount = list(projects_df['totalBackedAmount'])
corpNo = list(projects_df['corpNo'])

In [ ]:
# 최종 df 만들기
daily_supporter_df = pd.DataFrame()

for i in tqdm(range(len(campaignId))):
    
    # 프로젝트 1개를 하나의 df로 불러오기 (+ 인덱스 재정렬)
    tmp_df = supporter_df[supporter_df['campaignId'] == campaignId[i]]
    tmp_df = tmp_df.reset_index()
    tmp_df = tmp_df.drop('index', axis=1)
    
    # '날짜' 칼럼을 데이트타임 타입으로 형 변환
    tmp_df['whenCreated'] = pd.to_datetime(tmp_df['whenCreated'])
    tmp_df['whenCreatedDay'] = tmp_df['whenCreated'].dt.strftime('%Y-%m-%d')   # %Y-%m-%d으로 형식 포맷팅
    
    # 날짜별 Backer 수, Facebook 지지서명 수, 총 서포터 수 칼럼 추가를 위한 df 생성
    whenCreatedDay_df = pd.DataFrame(tmp_df.groupby('whenCreatedDay').count().index)
    
    # Backer 수, Facebook 지지서명 수, 총 서포터 수 칼럼 생성
    whenCreatedDay_df['totalBackerCnt'] = 0
    whenCreatedDay_df['totalFacebookCnt'] = 0
    whenCreatedDay_df['totalSupporterCnt'] = 0

    for c in range(len(whenCreatedDay_df)):
        totalBackerCnt = len(tmp_df[(tmp_df['type'] == 'B') & (tmp_df['whenCreatedDay'] == whenCreatedDay_df['whenCreatedDay'][c])])
        totalFacebookCnt = len(tmp_df[(tmp_df['type'] == 'S') & (tmp_df['whenCreatedDay'] == whenCreatedDay_df['whenCreatedDay'][c])])

        whenCreatedDay_df['totalBackerCnt'][c] = totalBackerCnt
        whenCreatedDay_df['totalFacebookCnt'][c] = totalFacebookCnt

    whenCreatedDay_df['totalSupporterCnt'] = whenCreatedDay_df['totalBackerCnt'] + whenCreatedDay_df['totalFacebookCnt']
    
    # 후원 금액 칼럼 만들기
    tmp_df2 = tmp_df
    
    # 공개하지 않은 후원 금액 제외한 일별 후원 금액 칼럼 만들기
    tmp_df2_O = tmp_df2[(tmp_df2['backedAmount'] != '###') & (tmp_df2['type'] == 'B')]
    tmp_df2_O = tmp_df2_O.reset_index()
    tmp_df2_O = tmp_df2_O.drop('index', axis=1)
    
    # 후원 금액 칼럼을 정수로 형 변환
    tmp_df2_O['backedAmount'] = tmp_df2_O['backedAmount'].astype('int')
    # 날짜별 각 칼럼 합계를 groupby
    tmp_df2_O = pd.DataFrame(tmp_df2_O.groupby('whenCreatedDay').sum())
    # backedAmount 칼럼만 추출
    tmp_df2_O = tmp_df2_O[['backedAmount']]
    
    # 일별 후원자 수 df와 일별 후원 금액 df 합치기
    whenCreatedDay_df = pd.merge(whenCreatedDay_df,tmp_df2_O, how='outer',on='whenCreatedDay')
    # backedAmount 결측치 0으로 대체
    whenCreatedDay_df['backedAmount'] = whenCreatedDay_df['backedAmount'].fillna(0)
    
    # 공개하지 않은 후원 금액 평균 낸 일별 후원 금액 칼럼 만들기
    tmp_df2_X = tmp_df2[(tmp_df2['backedAmount'] == '###') & (tmp_df2['type'] == 'B')]
    tmp_df2_X = tmp_df2_X.reset_index()
    tmp_df2_X = tmp_df2_X.drop('index', axis=1)
    
    # 날짜별 각 칼럼 수를 groupby
    tmp_df2_X = pd.DataFrame(tmp_df2_X.groupby('whenCreatedDay').count())
    # backedAmount 칼럼만 추출
    tmp_df2_X = tmp_df2_X[['backedAmount']]
    
    # 프로젝트 후원 금액 관련 변수 선언
    projectBackedAmount = totalBackedAmount[i]
    visibleBackedAmount = whenCreatedDay_df['backedAmount'].sum()
    hiddenBackedAmount = projectBackedAmount - visibleBackedAmount
    additional_avg = hiddenBackedAmount / tmp_df2_X['backedAmount'].sum()
    
    # 일별 확인 불가능한 후원 금액 평균 계산해서 열 추가
    tmp_df2_X['predictedAdditionalBackedAmount'] = round(tmp_df2_X['backedAmount'] * additional_avg)
    
    # 칼럼명 변경
    tmp_df2_X.rename(columns = {'backedAmount':'hiddenBackerCnt'}, inplace = True)
    
    # 일별 후원자 수 df와 일별 후원 금액 df 합치기
    whenCreatedDay_df = pd.merge(whenCreatedDay_df,tmp_df2_X, how='outer',on='whenCreatedDay')
    
    # backedAmount 결측치 0으로 대체
    whenCreatedDay_df['hiddenBackerCnt'] = whenCreatedDay_df['hiddenBackerCnt'].fillna(0)
    whenCreatedDay_df['predictedAdditionalBackedAmount'] = whenCreatedDay_df['predictedAdditionalBackedAmount'].fillna(0)
    
    # predictedBackedAmount 칼럼 만들기
    whenCreatedDay_df['predictedBackedAmount'] = whenCreatedDay_df['backedAmount'] + whenCreatedDay_df['predictedAdditionalBackedAmount']
    
    # 한 프로젝트의 확인 가능한 후원 금액, 확인 불가능한 후원 금액, (확인 불가능한 후원 금액) / (확인 불가능한 후원자 수)를 칼럼으로 추가
    whenCreatedDay_df['sumVisibleBackedAmount'] = visibleBackedAmount
    whenCreatedDay_df['sumHiddenBackedAmount'] = hiddenBackedAmount
    whenCreatedDay_df['avgHiddenBackedAmount'] = round(additional_avg) if math.isfinite(additional_avg) else 0
    
    # 메이커 id, 프로젝트 id 칼럼으로 추가
    whenCreatedDay_df.insert(0, 'corpNo', corpNo[i])
    whenCreatedDay_df.insert(1, 'campaignId', campaignId[i])
    
    # 최종 df에 프로젝트 하나 df 추가
    daily_supporter_df = daily_supporter_df.append(whenCreatedDay_df, ignore_index = True)

daily_supporter_df

In [ ]:
# 칼럼명 변경
daily_supporter_df.rename(columns = {'whenCreatedDay':'Date', 'backedAmount':'visibleBackedAmount'}, inplace = True)

# 칼럼 순서 변경
daily_supporter_df = daily_supporter_df[['corpNo', 'campaignId', 'Date', 'totalBackerCnt', 'totalFacebookCnt', 'totalSupporterCnt', 'visibleBackedAmount', 'predictedBackedAmount', 'hiddenBackerCnt', 'avgHiddenBackedAmount', 'predictedAdditionalBackedAmount', 'sumVisibleBackedAmount', 'sumHiddenBackedAmount']]

daily_supporter_df

In [ ]:
# csv로 저장
daily_supporter_df.to_csv('20221120_Wadiz_AllProjects_Supporters_df_Preprocessing_vF.csv', index=False, encoding='utf-8-sig')